## **FindMyText**: Step-by-step example

**FindMyText** is a Python package which can be used to find out whether a given text appears, in part or in full, within a text corpus. In other words, the purpose of **FindMyText** is to efficiently detect _text containment_, and is specifically designed to satisfy the three following criteria:
 
1. The method should be *robust* to small discrepancies between the texts (due to e.g. text normalisation, formatting variants, etc.)
2. The method should *scale* to large, web-crawled corpora 
3. And it should focus on detecting occurrences of _shared text segments_, rather than mere similarity

A central use case for **FindMyText** is to determine whether a large corpus contains texts that may be copyrighted or breach licensing conditions. The tool builds upon existing techniques for document fingerprinting, but with a new approach to precisely detect the presence of fingerprint chains. 

To run this notebook, install the package with its demo extras from the repository root: `pip install -e ".[demo]"`.

### Getting the data

In this short example, we will use a collection of 50 Italian-language books from the Gutenberg Project. They can be directly loaded from HuggingFace:

In [ ]:
from datasets import load_dataset
corpus = load_dataset("efederici/gutenberg-it")
corpus = corpus["train"].select(range(50))  # Limit to first 50 books for demo purposes

Those books are about 300-400 pages in length on average:

In [ ]:
avg_chars = sum(len(text["text"]) for text in corpus) // len(corpus) # type:ignore
print(f"Average book length: {avg_chars:,} characters")

### Building the index

The first step of **FindMyText** is to extract the set of _fingerprints_ for each text of the corpus. Document fingerprints are essentially $k$-grams extracted from the text and then converted to hashes. The fingerprints are then stored in an inverted index, which maps every fingerprint to a list of (`doc_id`, `position`) pairs indicating where the fingerprint has been observed in the corpus. To avoid storing the entire set of fingerprints, **FindMyText** relies on a technique called *winnowing*, which only keeps a small, deterministic subset of them (by default, $k$-grams of 4 words are hashed and the smallest hash is kept in every window of 6 consecutive $k$-grams).

The corpus is expected as a JSONL file (optionally compressed with GZip or Zstandard) with one JSON object per line, holding at least a unique `id` and a `text`. Building the index is then a single call, which returns the index ready for use:

In [ ]:
import json
import findmytext

# Index the first 100,000 characters per book to keep this demo bounded.
with open("gutenberg_demo.jsonl", "w", encoding="utf-8") as output:
    for book in corpus:
        output.write(json.dumps({"id": str(book["text_id"]), "text": book["text"][:100_000]}) + "\n")
index = findmytext.build_index("gutenberg_demo.jsonl", "gutenberg_index", delete_existing=True)

The index now sits in the directory `gutenberg_index`, with the fingerprints and their exact locations in the first 100,000 characters of each book (the query excerpt we use below is within this indexed prefix). Fingerprinting and merging are spread over several processes; `n_workers` sets how many (by default, all CPUs available to the process), and `max_memory_gb` bounds the memory they use. The same build is also available from the command line: `python -m findmytext.index_builder index gutenberg_demo.jsonl gutenberg_index`.

The index can be reopened later without rebuilding it:

In [ ]:
index = findmytext.DiskBasedIndex("gutenberg_index")

The constructed index is designed for scalability and efficiency. In particular, it is disk-based (through the use of memory-mapped arrays), making it possible to work with large corpora for which the index of fingerprints cannot be stored in memory. 

For a corpus split over many files, one can also build one index per file (for instance as separate cluster jobs) and merge them afterwards into a single index. The merged index is itself a valid input for later merges, so an index can be extended with new material at any time:

```python
findmytext.build_index("part1.jsonl.zst", "partial/part1")
findmytext.build_index("part2.jsonl.zst", "partial/part2")
# ... then:
index = findmytext.merge_indexes("partial", "my_index")
```

### Detecting text containment

Let's now use our index to detect which texts are or are not included in the corpus. We'll start by taking a small excerpt of our corpus, here a few lines of Italian poetry:

In [ ]:
excerpt = str(corpus[5]["text"][5078:6287])
print(excerpt)

The most straightforward way to detect text containment is by counting the number of shared fingerprints between the query and the indexed documents. The more fingerprints they share, the more likely it is that the query text is contained within the document.

To do this detection, we first load a `NbSharedFingerprintsDetector`, which takes as input the index (or the path of its directory):

In [ ]:
baseline_detector = findmytext.NbSharedFingerprintsDetector(index)

The detection can then be performed by calling `search`, which takes a text as input and returns a dictionary mapping document IDs to containment scores. Those scores are *probabilities*: the probability that the query shares a passage of at least `min_fragment_length` characters (1000 by default) with the document. Documents whose probability falls below a small `min_probability` threshold are left out of the results:

In [ ]:
print("Detecting verbatim excerpt in corpus based on number of shared fingerprints")
scores = baseline_detector.search(excerpt)
print("Scores:", scores)

best_match_id = max(scores, key=scores.get)
print(f"Best match: {best_match_id}  (probability: {scores[best_match_id]:.3f})")

correct_doc_id = str(corpus[5]["text_id"])  # The correct document ID for the excerpt we are testing against
print("Correct document ID:", correct_doc_id)

Unsurprisingly, the containment probability for the correct document ID is very high, indicating that the excerpt is indeed very likely to be present in that document.

### Near-verbatim detection 

We can check what happens if there are small discrepancies between the excerpt and the text in the corpus. Those discrepancies are highly common, and may be due to text standardisation, presence of OCR or formatting errors, or boilerplate, etc. Here, we apply some changes to the spelling and remove the stanza and page numbers:

In [ ]:
import re

near_verbatim = (excerpt
    .replace("cc", "c")    # simplify doubled consonants: "Ricciardo" → "Riciardo"
    .replace("ss", "s")    # "stesso" → "steso", "appresso" → "appreso"
    .replace("gli", "li")  # archaic Italian spelling: "gli altri" → "li altri"
    .replace("gn", "ni")   # phonetic spelling of gn: "signore" → "siniore"
)

# Remove the stanza and page numbers standing on their own line
near_verbatim = re.sub(r"^\s*\[?\d+\]?\s*$", "", near_verbatim, flags=re.MULTILINE)

To derive the best alignment between this modified excerpt and the original book, one can use the ready-made local aligner implemented in <font face="Century Gothic, sans-serif"><b>Find<span style="color:#4A7A4A;">My</span>Text</span></b></font>, based on an efficient implementation of the Smith-Waterman algorithm : 

In [ ]:
alignment = findmytext.align(near_verbatim, str(corpus[5]["text"]))
alignment.show()

The detector remains robust to those changes, and still detects the excerpt as being contained in the original text, although with a slightly lower probability, as can be seen below:

In [ ]:
print("Detecting near-verbatim excerpt in corpus")
near_verbatim_scores = baseline_detector.search(near_verbatim)
print("Containment probabilities:", near_verbatim_scores)

However, the `NbSharedFingerprintsDetector` has a major flaw: its score only depends on the presence of shared fingerprints, but it ignores their _order_. This means that a text in which lines are jumbled will still receive a high probability, although the excerpt no longer represents a contiguous fragment of the original text: 

In [ ]:
print("Detecting excerpt with all lines shuffled")
import random
lines = excerpt.split("\n")
random.shuffle(lines)
jumbled_excerpt = "\n".join(lines)

jumbled_scores = baseline_detector.search(jumbled_excerpt)
print("Containment probabilities:", jumbled_scores)

This stands in sharp contrast with the outputs of the local aligner, which tells us that the excerpt is no longer well aligned to the original text:

In [ ]:
alignment = findmytext.align(jumbled_excerpt, str(corpus[5]["text"]))
alignment.show()

### Detecting fingerprint chains

Fortunately, this shortcoming can be addressed by explicitly searching for _chains_ of fingerprints that are shared between the excerpt and a document in the corpus. As explained in our paper, detecting the presence of such chains can be done through a clustering approach, which we implement in `FingerprintChainDetector`:

In [ ]:
chain_detector = findmytext.FingerprintChainDetector(index)

print("Detecting verbatim excerpt in corpus based on the detection of fingerprint chains:")
score_correct = chain_detector.search(excerpt)
print("Containment probabilities:", score_correct)

print("Detecting near-verbatim excerpt in corpus based on the detection of fingerprint chains:")
score_near = chain_detector.search(near_verbatim)
print("Containment probabilities:", score_near)

As we can see, the `FingerprintChainDetector` provides similar high containment probabilities for both the verbatim and near-verbatim excerpts. However, it is no longer fooled by texts in which fingerprints appear in a different order than the original text:

In [ ]:
print("Detecting shuffled excerpt in corpus based on the detection of fingerprint chains:")
score_shuffled = chain_detector.search(jumbled_excerpt)
print("Containment probabilities:", score_shuffled)

### Looking under the hood

`search` is a shortcut for two steps: `retrieve` fingerprints the query and fetches the candidate documents from the index, and `raw_scores` computes the score of each candidate before it is turned into a probability. For the `NbSharedFingerprintsDetector`, this raw score is simply the number of fingerprints shared with the query; for the `FingerprintChainDetector`, it is the number of fingerprints in the longest chain. Comparing both on the verbatim and shuffled excerpts shows what the probabilities above are based on:

In [ ]:
for name, text in [("verbatim", excerpt), ("shuffled", jumbled_excerpt)]:
    candidates = chain_detector.retrieve(text)
    print(f"{name} excerpt:")
    print("  shared fingerprints:       ", baseline_detector.raw_scores(candidates))
    print("  fingerprints in best chain:", chain_detector.raw_scores(candidates))

### Tuning the detector

A few options may be worth adjusting depending on the use case:

- `min_fragment_length` (an argument of `search`) sets the length in characters of the shared passage one wishes to detect. A smaller value makes the detector sensitive to shorter fragments, at the cost of more false positives.
- `rarity_weighted=True` gives more weight to rare fingerprints than to common phrases, which helps when the corpus contains many near-duplicates or boilerplate.
- `n_workers` sets the number of threads reading the index in parallel. This only pays off when the index sits on slow or network storage; the sequential default is faster otherwise.

In [ ]:
# Detect shorter shared passages (here, half of the default length)
print("Shorter fragments:", chain_detector.search(near_verbatim, min_fragment_length=500))

# Down-weight common phrases
weighted_detector = findmytext.FingerprintChainDetector(index, rarity_weighted=True)
print("Rarity-weighted: ", weighted_detector.search(near_verbatim))

# Parallel reads from the index (pass the index path rather than an opened index)
parallel_detector = findmytext.FingerprintChainDetector("gutenberg_index", n_workers=4)
print("Parallel reads:  ", parallel_detector.search(near_verbatim))

Contact us if you have any questions or would like to contribute to the project in one way or another. And do not hesitate to reach out and share how you are using **FindMyText** in your research or projects. We would love to hear from you :-)